# Instructor solution: Assignment 3 (RAG evaluation, City Library FAQ)
Run from the `evals-repo` folder (it needs `common.py` and `data/assignment_rag.json`). Live Gemini calls use `.env`.
Results are saved in `_instructor_private/`.

**Notes for the instructor**
- LLM outputs vary a little between runs, so a student's table may differ from yours. Grade the **code and the logic**, not exact numbers.
- Task 4 includes a clearly marked **fixture** (an answer written by hand, not by the model) so the faithfulness judge has a known failure to catch.
- The corpus has only 3 documents, so differences between grid settings are small. This is for practising the code.
- The code-check cell at the end tests the metric and grading functions **without any API call**.

In [ ]:
from common import ask, embed, MODEL
import json, math, re
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
assert (ROOT / "common.py").exists(), "Run this notebook from evals-repo."
data = json.loads((ROOT / "data" / "assignment_rag.json").read_text(encoding="utf-8"))
docs, queries = data["documents"], data["queries"]
PRIVATE = ROOT / "_instructor_private"
PRIVATE.mkdir(exist_ok=True)
K = 3
print("Model:", MODEL, "|", len(docs), "documents |", len(queries), "queries")

## Task 1: chunks and relevance grades

In [ ]:
def norm(text):
    return " ".join(text.lower().split())

def make_chunks(documents, size, overlap):
    if not 0 <= overlap < size:
        raise ValueError("Need 0 <= overlap < size")
    step = size - overlap
    chunks = []
    for doc in documents:
        words = doc["text"].split()
        start = 0
        while True:
            chunks.append({"chunk_id": len(chunks), "doc_id": doc["id"], "text": " ".join(words[start:start + size])})
            if start + size >= len(words):
                break
            start += step
    return chunks

def grade(chunk, query):
    if norm(query["answer_span"]) in norm(chunk["text"]):
        return 2
    return 1 if chunk["doc_id"] == query["doc_id"] else 0

# Every answer span must be in its own document (and only there).
for q in queries:
    owners = [d["id"] for d in docs if norm(q["answer_span"]) in norm(d["text"])]
    assert owners == [q["doc_id"]], (q["query"], owners)

chunks_30 = make_chunks(docs, 30, 0)
print("size=30, overlap=0 ->", len(chunks_30), "chunks")
print("Relevant chunks per query:", [sum(grade(c, q) == 2 for c in chunks_30) for q in queries])

## Task 2: retrieval metrics and embedding retrieval

In [ ]:
def hit_at_k(grades, k, thr=2):
    return float(any(g >= thr for g in grades[:k]))

def reciprocal_rank(grades, k, thr=2):
    for rank, g in enumerate(grades[:k], start=1):
        if g >= thr:
            return 1 / rank
    return 0.0

def dcg_at_k(grades, k):
    return sum(g / math.log2(rank + 1) for rank, g in enumerate(grades[:k], start=1))

def ndcg_at_k(grades, k):
    ideal = dcg_at_k(sorted(grades[:k], reverse=True), k)
    return dcg_at_k(grades, k) / ideal if ideal > 0 else 0.0

test = [0, 2, 0, 1]
print("Test on", test, "-> Hit@3:", hit_at_k(test, 3), "| RR:", reciprocal_rank(test, 3), "| NDCG@3:", round(ndcg_at_k(test, 3), 2))

_cache = {}
def embed_texts(texts):
    todo = [t for t in dict.fromkeys(texts) if t not in _cache]
    for i in range(0, len(todo), 50):
        part = todo[i:i + 50]
        for text, vec in zip(part, embed(part)):
            _cache[text] = np.array(vec, dtype=float)
    m = np.array([_cache[t] for t in texts])
    return m / np.linalg.norm(m, axis=1, keepdims=True)

def evaluate(size, overlap, k=K):
    chunks = make_chunks(docs, size, overlap)
    chunk_vecs = embed_texts([c["text"] for c in chunks])
    rows = []
    for q in queries:
        scores = chunk_vecs @ embed_texts([q["query"]])[0]
        order = [int(i) for i in np.argsort(-scores, kind="stable")]
        ranked = [grade(chunks[i], q) for i in order]
        rows.append({"query": q["query"], "hit": hit_at_k(ranked, k),
                     "rr": reciprocal_rank(ranked, k), "ndcg": ndcg_at_k(ranked, k)})
    return chunks, pd.DataFrame(rows)

_, df_30 = evaluate(30, 0)
print("size=30, overlap=0:", {"Hit@3": round(float(df_30.hit.mean()), 2), "MRR@3": round(float(df_30.rr.mean()), 2), "NDCG@3": round(float(df_30.ndcg.mean()), 2)})

## Task 3: grid search

In [ ]:
rows = []
for size in [30, 60]:
    for overlap in [0, 15]:
        chunks, df = evaluate(size, overlap)
        rows.append({"chunk_size": size, "overlap": overlap, "n_chunks": len(chunks),
                     "Hit@3": df.hit.mean(), "MRR@3": df.rr.mean(), "NDCG@3": df.ndcg.mean()})
grid = pd.DataFrame(rows).round(2)
display(grid)
best = grid.sort_values(["Hit@3", "MRR@3"], ascending=False).iloc[0]
BEST_SIZE, BEST_OVERLAP = int(best["chunk_size"]), int(best["overlap"])
print(f"Best setting: size={BEST_SIZE}, overlap={BEST_OVERLAP}")

## Task 4: generation, faithfulness judge, and diagnosis

In [ ]:
GROUNDED = ("Answer the question using ONLY the documents provided. If they do not contain the answer, "
            "reply exactly: I don't know based on the documents. Answer in one or two sentences.")

best_chunks = make_chunks(docs, BEST_SIZE, BEST_OVERLAP)
best_vecs = embed_texts([c["text"] for c in best_chunks])

def top_chunks(query, k=K):
    scores = best_vecs @ embed_texts([query])[0]
    return [best_chunks[int(i)] for i in np.argsort(-scores, kind="stable")[:k]]

def answer_with(query, context_text, system):
    return ask(f"Documents:\n{context_text}\n\nQuestion: {query}", system=system, temperature=0).strip()

FAITH_PROMPT = """You check whether an answer is faithful to the documents it was based on.
PASS: every claim in the answer is supported by the documents.
FAIL: the answer contains a claim that is not in the documents, distorts what they say, or uses outside knowledge.

Documents:
{context}

Question: {query}
Answer: {answer}

Reply as JSON: {"reasoning": "one short sentence", "answer": "Pass" or "Fail"}"""

def judge_faithfulness(context, query, answer):
    out = ask(FAITH_PROMPT.replace("{context}", context).replace("{query}", query).replace("{answer}", answer), as_json=True)
    if isinstance(out, list):
        out = out[0]
    verdict = str(out.get("answer", "")).strip().capitalize()
    if verdict not in ("Pass", "Fail"):
        raise ValueError(f"Judge must answer Pass or Fail, got: {out}")
    return verdict == "Pass"

def keyword_correct(answer, keywords):
    return any(k.lower() in answer.lower() for k in keywords)

def diagnose(hit, correct):
    if hit and correct:     return "OK"
    if hit and not correct: return "generation problem"
    if correct:             return "lucky"
    return "retrieval problem"

def run_row(q, context_chunks, answer, is_fixture=False):
    context = "\n\n".join(c["text"] for c in context_chunks)
    hit = any(norm(q["answer_span"]) in norm(c["text"]) for c in context_chunks)
    correct = keyword_correct(answer, q["answer_keywords"])
    return {"query": q["query"], "retrieval_hit": hit, "answer_correct": correct,
            "faithful": judge_faithfulness(context, q["query"], answer),
            "diagnosis": diagnose(hit, correct), "is_fixture": is_fixture, "answer": answer}

rows = []
for q in queries:
    ctx = top_chunks(q["query"])
    rows.append(run_row(q, ctx, answer_with(q["query"], "\n\n".join(c["text"] for c in ctx), GROUNDED)))

# FIXTURE: an answer written by hand (NOT produced by the model) to give the judge a known failure.
fixture_q = queries[0]
fixture_answer = "You can borrow up to 15 books at a time, and the library delivers them to your home."
rows.append(run_row(fixture_q, top_chunks(fixture_q["query"]), fixture_answer, is_fixture=True))

results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
display(results[["query", "retrieval_hit", "answer_correct", "faithful", "diagnosis", "is_fixture"]])
print(results["diagnosis"].value_counts().to_string())
print("\nFixture row, judged faithful?", bool(results.iloc[-1]["faithful"]), "(expected: False, because the answer invents '15 books' and home delivery)")

## Task 5: context swap

In [ ]:
swap_q = next(q for q in queries if "Sunday" in q["query"])
contexts = {"real document": next(d["text"] for d in docs if d["id"] == swap_q["doc_id"]),
            "unrelated document": next(d["text"] for d in docs if d["id"] == "events")}
systems = {"plain prompt": "You are a helpful library assistant.", "grounded prompt": GROUNDED}

rows = []
for sys_name, system in systems.items():
    for ctx_name, ctx in contexts.items():
        rows.append({"prompt": sys_name, "context": ctx_name, "answer": answer_with(swap_q["query"], ctx, system)})
swap = pd.DataFrame(rows)
print("Question:", swap_q["query"])
display(swap)
print("Good behaviour: real document -> 'closed on Sundays'; unrelated document -> 'I don't know'.")

In [ ]:
(PRIVATE / "assignment3_results.json").write_text(json.dumps(
    {"model": MODEL, "best_setting": {"size": BEST_SIZE, "overlap": BEST_OVERLAP},
     "grid": grid.to_dict("records"), "generation": results.to_dict("records"),
     "context_swap": swap.to_dict("records")}, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)), encoding="utf-8")
print("Saved", PRIVATE / "assignment3_results.json")

## Code checks (no API calls)
These assertions test the grading and metric functions on their own.

In [ ]:
# make_chunks
assert len(make_chunks(docs, 30, 0)) == 12
assert make_chunks(docs[:1], 20, 10)[1]["text"].split()[:10] == make_chunks(docs[:1], 20, 10)[0]["text"].split()[10:20]
assert all(c["text"].strip() for c in make_chunks(docs, 60, 15))          # no empty chunks
for bad in [(20, 20), (20, 25)]:
    try:
        make_chunks(docs, *bad)
    except ValueError:
        pass
    else:
        raise AssertionError("overlap >= size was accepted")

# grade
q = queries[3]                                                             # Sunday question
fake_hit = {"doc_id": q["doc_id"], "text": "The library is   CLOSED on Sundays and public holidays."}
fake_same = {"doc_id": q["doc_id"], "text": "Study rooms fit 6 people."}
fake_other = {"doc_id": "events", "text": "Story hour is on Wednesday."}
assert (grade(fake_hit, q), grade(fake_same, q), grade(fake_other, q)) == (2, 1, 0)

# metrics
assert hit_at_k([0, 2, 0, 1], 3) == 1.0 and hit_at_k([0, 1, 1, 2], 3) == 0.0
assert reciprocal_rank([0, 2, 0, 1], 3) == 0.5 and reciprocal_rank([0, 0, 0, 2], 3) == 0.0
assert round(ndcg_at_k([0, 2, 0, 1], 3), 2) == 0.63
assert ndcg_at_k([2, 1, 0], 3) == 1.0 and ndcg_at_k([0, 0, 0], 3) == 0.0
assert ndcg_at_k([1, 1, 0], 3) == 1.0          # perfect order of related-only chunks: why NDCG needs Hit/Recall next to it

# generation helpers
assert keyword_correct("The late fee is $0.25 per day.", ["0.25"]) and not keyword_correct("I don't know.", ["0.25"])
assert diagnose(True, True) == "OK" and diagnose(True, False) == "generation problem"
assert diagnose(False, False) == "retrieval problem" and diagnose(False, True) == "lucky"
print("All code checks passed.")